<a href="https://colab.research.google.com/github/Subhranshu-123/BIKE2/blob/main/LSTM(PRACTICE).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PREPARE DATA

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
data = [
    ("i love this movie", 1),
    ("this film was terrible", 0),
    ("great acting and story", 1),
    ("i hated the plot", 0),
    ("highly recommended masterpiece", 1),
    ("boring waste of time", 0)
]


In [3]:
vocab = {"<PAD>": 0}  # Use a padding token for empty spaces if needed
for phrase, _ in data:
    for word in phrase.split():
        if word not in vocab:
            vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)
print(f"Vocabulary Size: {VOCAB_SIZE}")

Vocabulary Size: 22


Numericalize and Pad Sequences

In [14]:
# Find the maximum phrase length to handle padding
max_len = max(len(phrase.split()) for phrase, _ in data)

# Pseudo-code translation

Initialize X_data as an empty list
Initialize y_data as an empty list

For each (phrase, label) pair in the dataset:
    Initialize an empty list called tokens
    
    For each word in the phrase:
        Find the numerical index of the word in the vocabulary
        Add this numerical index to the tokens list
        
    Calculate how many padding tokens are needed (max_len minus length of tokens)
    Multiply the "<PAD>" index by that number
    Append those padding indexes to the end of the tokens list
    
    Append the complete tokens list to X_data
    Append the label to y_data


In [5]:
X_data, y_data = [], []
for phrase, label in data:
    # Convert words to vocabulary indices
    tokens = [vocab[word] for word in phrase.split()]
    # Pad shorter sequences with zeros at the end
    tokens += [vocab["<PAD>"]] * (max_len - len(tokens))
    X_data.append(tokens)
    y_data.append(label)

In [6]:
# Convert lists to PyTorch Tensors
X_tensor = torch.tensor(X_data, dtype=torch.long)
y_tensor = torch.tensor(y_data, dtype=torch.float32)

LSTM ARCHITECTURE



Define a class named SimpleTextLSTM that inherits from the Neural Network Module:

    Method Initialize(vocab_size, embedding_dim, hidden_dim, output_dim):
        Call the parent class initialization
        Create an Embedding layer (inputs: vocab_size, outputs: embedding_dim)
        Create an LSTM layer (inputs: embedding_dim, outputs: hidden_dim, set batch_first to True)
        Create a Linear layer (inputs: hidden_dim, outputs: output_dim)

    Method Forward(text):
        Pass the text through the Embedding layer to get 'embedded'
        Pass 'embedded' through the LSTM layer to get 'lstm_out', 'hidden', and 'cell'
        
        Extract the last layer's final hidden state ('final_hidden') from the 'hidden' tensor
        Pass 'final_hidden' through the Linear layer
        Return the result


In [7]:
class SimpleTextLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SimpleTextLSTM, self).__init__()
        # Dense vector representation layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # Core LSTM layer processing word sequences sequentially
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        # Linear layer mapping hidden states to output class scores
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        lstm_out, (hidden, cell) = self.lstm(embedded)

        # Take the final hidden state of the sequence to make a prediction
        final_hidden = hidden[-1, :, :] # shape: [batch_size, hidden_dim]
        return self.fc(final_hidden)


Initialize and Train the Model

In [8]:
# Hyperparameters
EMBEDDING_DIM = 8
HIDDEN_DIM = 16
OUTPUT_DIM = 1  # Single logit output for binary classification

In [9]:
model = SimpleTextLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM)
criterion = nn.BCEWithLogitsLoss()  # Binary cross entropy optimized for raw outputs
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [10]:
# Training loop
model.train()
for epoch in range(20):  # Run for 20 iterations
    optimizer.zero_grad()

    # Forward pass
    predictions = model(X_tensor).squeeze(1)
    loss = criterion(predictions, y_tensor)

    # Backward pass and weight updates
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | Loss: {loss.item():.4f}")

Epoch 05 | Loss: 0.6472
Epoch 10 | Loss: 0.5339
Epoch 15 | Loss: 0.3518
Epoch 20 | Loss: 0.1702


Evaluate and Run Inference

In [11]:
model.eval()
with torch.no_grad():
    # New query text phrase
    test_phrase = "i love acting"
    # Convert token targets with default safe fallback 0 for out-of-vocab words
    test_tokens = [vocab.get(word, 0) for word in test_phrase.split()]
    # Pad to max length
    test_tokens += [vocab["<PAD>"]] * (max_len - len(test_tokens))

In [13]:
# Format sample into batch size of 1
input_tensor = torch.tensor([test_tokens], dtype=torch.long)

# Compute output and squeeze probabilities through Sigmoid function
raw_output = model(input_tensor)
probability = torch.sigmoid(raw_output).item()

sentiment = "Positive" if probability >= 0.5 else "Negative"
print(f"\nInference on text: '{test_phrase}'")
print(f"Confidence Score: {probability:.4f} -> Resulting Sentiment: {sentiment}")


Inference on text: 'i love acting'
Confidence Score: 0.9097 -> Resulting Sentiment: Positive
